In [2]:
"""
Personalized training  +  FEATURE METHOD 2: concatenate-after-encoder.

Each cycle is time-normalized to TARGET_LEN points and fed to an LSTM/GRU/CNN
encoder. Two scalar gait features (cadence, duration variability) enter through
a SECOND input branch and are concatenated onto the encoder output before the
final dense layer (Keras functional API).

Set USE_FEATURES = False to get the sequence-only baseline (single input).

Leak-safety: duration variability is computed on the TRAIN cycles of each
subject only; cadence is per-cycle; scalars standardized with train statistics.
"""
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from scipy.interpolate import interp1d
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, LSTM, GRU, Conv1D, GlobalAveragePooling1D,
                                     Dense, Dropout, Concatenate)
from tensorflow.keras.callbacks import EarlyStopping

# =========================
# Configuration
# =========================
ROOT_DIR = 'All_10person_Cycles'
TARGET_LEN = 100
SAMPLING_RATE = 100.0
TRAIN_RATIO = 0.8
VALID_LABELS = ['back', 'front', 'normal', 'side']

USE_FEATURES = False        # <-- toggle for ablation

IMU_COLUMNS = [
    'IMU101_v0', 'IMU101_v1', 'IMU103_v0', 'IMU103_v1',
    'IMU104_v0', 'IMU104_v1', 'IMU301_v0', 'IMU301_v1',
]
PRESSURE_COLUMNS = [
    'x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7',
    'x8', 'x9', 'x10', 'x11', 'x12', 'x13', 'x14', 'x15'
]
CONFIGS = {
    'IMU':          IMU_COLUMNS,
    'Pressure':     PRESSURE_COLUMNS,
    'IMU+Pressure': IMU_COLUMNS + PRESSURE_COLUMNS,
}

UNITS = 64
DROPOUT_RATE = 0.3
EPOCHS = 25
BATCH_SIZE = 32
MODEL_TYPES = ['lstm', 'gru', 'cnn']

RESULTS_DIR = 'gait_hardware_paper_results'
tag = 'featconcat' if USE_FEATURES else 'seqonly_concat'
RESULTS_CSV = os.path.join(RESULTS_DIR, f'personalized_{tag}_results.csv')


def get_subject_id(file_path):
    return os.path.relpath(file_path, ROOT_DIR).split(os.sep)[0]


def resample_cycle(features, target_len=TARGET_LEN):
    n = features.shape[0]
    kind = 'cubic' if n >= 4 else 'linear'
    old_t = np.linspace(0, 1, n)
    new_t = np.linspace(0, 1, target_len)
    return interp1d(old_t, features, axis=0, kind=kind)(new_t)


def load_all_raw():
    raw = []
    if not os.path.exists(ROOT_DIR):
        print(f"ROOT_DIR not found: {ROOT_DIR}")
        return raw
    for root, dirs, files in os.walk(ROOT_DIR):
        if root.endswith('Abnormal'):
            for filename in files:
                if not filename.endswith('.csv'):
                    continue
                fp = os.path.join(root, filename)
                parts = filename.replace('.csv', '').split('__')
                if len(parts) != 2:
                    continue
                name_label_part, raw_id_part = parts
                label = name_label_part.split('_')[-1]
                if raw_id_part.count('_') > 1:
                    continue
                if raw_id_part.count('_') == 1 and not raw_id_part.startswith('cycle_'):
                    continue
                cid = raw_id_part.split('_')[-1]
                if label not in VALID_LABELS or not cid.isdigit():
                    continue
                try:
                    df = pd.read_csv(fp)
                except Exception:
                    continue
                raw.append((df, label, get_subject_id(fp)))
    return raw


def prepare_subject_matrix(subject_rows, feature_cols):
    X, y, nrows = [], [], []
    for df, label, _ in subject_rows:
        if any(c not in df.columns for c in feature_cols):
            continue
        vals = df[feature_cols].values
        if vals.shape[0] < 2 or vals.shape[1] == 0:
            continue
        nrows.append(vals.shape[0])
        X.append(resample_cycle(vals))
        y.append(label)
    if len(X) == 0:
        return None, None, None
    return np.array(X), np.array(y), np.array(nrows)


def build_model(model_type, seq_len, num_features, num_scalars, num_classes, use_features):
    """Functional model. If use_features, a second scalar input is concatenated
    after the sequence encoder; otherwise it's a single-input model."""
    seq_in = Input(shape=(seq_len, num_features), name='sequence')
    if model_type == 'lstm':
        x = LSTM(UNITS, name='LSTM')(seq_in)
        x = Dropout(DROPOUT_RATE)(x)
    elif model_type == 'gru':
        x = GRU(UNITS, name='GRU')(seq_in)
        x = Dropout(DROPOUT_RATE)(x)
    elif model_type == 'cnn':
        x = Conv1D(64, 5, activation='relu')(seq_in)
        x = Dropout(DROPOUT_RATE)(x)
        x = Conv1D(64, 5, activation='relu')(x)
        x = GlobalAveragePooling1D()(x)
        x = Dropout(DROPOUT_RATE)(x)
    else:
        raise ValueError(model_type)

    if use_features:
        scal_in = Input(shape=(num_scalars,), name='scalars')
        x = Concatenate()([x, scal_in])
        out = Dense(num_classes, activation='softmax')(x)
        model = Model(inputs=[seq_in, scal_in], outputs=out)
    else:
        out = Dense(num_classes, activation='softmax')(x)
        model = Model(inputs=seq_in, outputs=out)

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


def run():
    raw = load_all_raw()
    if not raw:
        print("No cycles loaded.")
        return
    subjects = sorted({r[2] for r in raw})
    print(f"Detected {len(subjects)} subjects: {subjects}")
    print(f"USE_FEATURES = {USE_FEATURES}\n")

    rows = []
    for subj in subjects:
        subj_rows = [r for r in raw if r[2] == subj]
        print(f"\n{'='*55}\n=== Subject: {subj} ({len(subj_rows)} cycles) ===\n{'='*55}")

        for cfg_name, cols in CONFIGS.items():
            X, y_str, nrows = prepare_subject_matrix(subj_rows, cols)
            if X is None:
                print(f"  [{cfg_name}] no usable cycles, skipping")
                continue

            le = LabelEncoder()
            y_int = le.fit_transform(y_str)
            num_classes = len(le.classes_)
            y_cat = to_categorical(y_int, num_classes=num_classes)

            n = X.shape[0]
            uniq, counts = np.unique(y_int, return_counts=True)
            if n < 10 or counts.min() < 2 or num_classes < 2:
                print(f"  [{cfg_name}] too few cycles (n={n}, min/class={counts.min()}), skipping")
                continue

            idx = np.arange(n)
            try:
                tr_idx, te_idx = train_test_split(idx, train_size=TRAIN_RATIO,
                                                  random_state=42, shuffle=True, stratify=y_int)
            except ValueError:
                tr_idx, te_idx = train_test_split(idx, train_size=TRAIN_RATIO,
                                                  random_state=42, shuffle=True)

            cadence = SAMPLING_RATE / nrows
            dur_std_train = float(np.std(nrows[tr_idx]))
            scalar = np.column_stack([cadence, np.full(n, dur_std_train)])
            num_scalars = scalar.shape[1]

            if USE_FEATURES:
                scaler = StandardScaler().fit(scalar[tr_idx])
                scalar = scaler.transform(scalar)

            seq_len = X.shape[1]
            num_features = X.shape[2]
            Xtr_s, Xte_s = X[tr_idx], X[te_idx]
            sctr, scte = scalar[tr_idx], scalar[te_idx]
            ytr, yte = y_cat[tr_idx], y_cat[te_idx]
            yte_int = y_int[te_idx]

            for mtype in MODEL_TYPES:
                tf.keras.backend.clear_session()
                model = build_model(mtype, seq_len, num_features, num_scalars,
                                    num_classes, USE_FEATURES)
                train_x = [Xtr_s, sctr] if USE_FEATURES else Xtr_s
                test_x = [Xte_s, scte] if USE_FEATURES else Xte_s
                model.fit(train_x, ytr, epochs=EPOCHS, batch_size=BATCH_SIZE,
                          callbacks=[EarlyStopping(monitor='loss', patience=5,
                                                   restore_best_weights=True, verbose=0)],
                          verbose=0)
                pred = np.argmax(model.predict(test_x, verbose=0), axis=1)
                acc = accuracy_score(yte_int, pred)
                print(f"  [{cfg_name:12s} {mtype.upper():4s}] "
                      f"n_test={len(yte_int):3d}  acc={acc:.4f}")
                rows.append({
                    'subject': subj, 'config': cfg_name, 'model': mtype,
                    'use_features': USE_FEATURES, 'feature_method': 'concat',
                    'n_cycles': n, 'n_test': len(yte_int), 'n_classes': num_classes,
                    'accuracy': round(acc, 4),
                    'macro_f1': round(f1_score(yte_int, pred, average='macro', zero_division=0), 4),
                    'weighted_f1': round(f1_score(yte_int, pred, average='weighted', zero_division=0), 4),
                })

    if not rows:
        print("\nNo results produced.")
        return
    os.makedirs(RESULTS_DIR, exist_ok=True)
    df = pd.DataFrame(rows)
    df.to_csv(RESULTS_CSV, index=False)
    print(f"\n\nWrote {len(df)} rows to {RESULTS_CSV}")
    with pd.option_context('display.max_rows', None, 'display.width', 120):
        print(df.pivot_table(index='subject', columns=['config', 'model'], values='accuracy'))


if __name__ == '__main__':
    tf.get_logger().setLevel('ERROR')
    run()

Detected 10 subjects: ['Andy_Dynamic', 'Ankan_Dynamic', 'David_Dynamic', 'Hrithik_Dynamic', 'JJ_Dynamic', 'Mustafa_Dynamic', 'Rezoan_Dynamic', 'Sudipta_Dynamic', 'Tianjun_Dynamic', 'Zongwei_Dynamic']
USE_FEATURES = False


=== Subject: Andy_Dynamic (424 cycles) ===
  [IMU          LSTM] n_test= 85  acc=0.9294
  [IMU          GRU ] n_test= 85  acc=1.0000
  [IMU          CNN ] n_test= 85  acc=0.9882
  [Pressure     LSTM] n_test= 85  acc=0.9882
  [Pressure     GRU ] n_test= 85  acc=0.9647
  [Pressure     CNN ] n_test= 85  acc=0.9412
  [IMU+Pressure LSTM] n_test= 85  acc=0.9647
  [IMU+Pressure GRU ] n_test= 85  acc=0.9647
  [IMU+Pressure CNN ] n_test= 85  acc=0.9882

=== Subject: Ankan_Dynamic (528 cycles) ===
  [IMU          LSTM] n_test=106  acc=0.9906
  [IMU          GRU ] n_test=106  acc=0.9717
  [IMU          CNN ] n_test=106  acc=0.9906
  [Pressure     LSTM] n_test=106  acc=0.9717
  [Pressure     GRU ] n_test=106  acc=0.8774
  [Pressure     CNN ] n_test=106  acc=0.9623
  [IMU+Pressur